# k-means and PCA from scratch (numpy only)

This notebook goes with `notes.md` §8 (k-means) and §9 (PCA). It runs Lloyd's
algorithm and checks that the k-means objective never increases, then computes
PCA from the eigendecomposition of the covariance matrix, checks it against
sklearn and against the SVD, and verifies that the reconstruction error with
$k$ components equals $(n-1)$ times the sum of the discarded eigenvalues.

All randomness comes from fixed seeds, and the notebook runs in a few seconds
on a CPU.

In [1]:
import numpy as np

np.random.seed(42)

## 1. Lloyd's algorithm and its monotone objective (notes §8.1-§8.2)

The data is three 2-D Gaussian blobs of 300 points each (standard deviation
1.1, centers $(0, 0)$, $(4, 1)$, $(2, 3.5)$), close enough to overlap so that
the algorithm needs several iterations. Each iteration does the two exact
minimizations from the notes: assign every point to its nearest center, then
move every center to the mean of its points. The inertia
$J = \sum_i \|x_i - \mu_{z_i}\|^2$ is recorded after each iteration, and the
assert checks that it never increases. Purity (the fraction of points whose
cluster's majority true label matches their own) should be high but below 1
because the blobs overlap.

In [2]:
def kmeans(X, k, n_iters=100, seed=0):
    # Lloyd's algorithm. Returns (centers, labels, inertia_trace).
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=k, replace=False)].copy()  # k data points
    trace = []
    labels = None
    for _ in range(n_iters):
        # Assignment step: nearest center for every point.
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)  # (n, k)
        new_labels = d2.argmin(axis=1)
        # Update step: each center moves to the mean of its points.
        for j in range(k):
            mask = new_labels == j
            if mask.any():
                centers[j] = X[mask].mean(axis=0)
        trace.append(float(((X - centers[new_labels]) ** 2).sum()))
        if labels is not None and np.array_equal(labels, new_labels):
            break  # assignments unchanged: converged
        labels = new_labels
    return centers, labels, trace


n_per = 300
X_km = np.vstack([
    np.random.randn(n_per, 2) * 1.1 + np.array([0.0, 0.0]),
    np.random.randn(n_per, 2) * 1.1 + np.array([4.0, 1.0]),
    np.random.randn(n_per, 2) * 1.1 + np.array([2.0, 3.5]),
])
true_labels = np.repeat(np.arange(3), n_per)

centers, labels, trace = kmeans(X_km, k=3, seed=7)
print(f"converged in {len(trace)} iterations; inertia trace:")
for i, v in enumerate(trace):
    print(f"  iter {i:2d}: inertia = {v:12.4f}")
assert np.all(np.diff(trace) <= 1e-9), "inertia increased"
print("PASS: inertia never increases.")

purity = sum(np.bincount(true_labels[labels == j]).max()
             for j in range(3) if np.any(labels == j)) / len(X_km)
print(f"cluster purity vs. ground truth = {purity:.4f}")
print(f"final centers:\n{np.round(centers, 3)}")

converged in 5 iterations; inertia trace:
  iter  0: inertia =    3277.8670
  iter  1: inertia =    2006.2492
  iter  2: inertia =    1889.5045
  iter  3: inertia =    1881.5669
  iter  4: inertia =    1881.5669
PASS: inertia never increases.
cluster purity vs. ground truth = 0.9311
final centers:
[[ 2.04   3.68 ]
 [ 4.194  1.   ]
 [-0.014 -0.062]]


## 2. Check against sklearn's KMeans

sklearn runs k-means++ initialization with 10 restarts and keeps the best.
Our single run from one random initialization should reach an inertia within
5% of sklearn's, since the blobs are well enough separated that most
initializations end at the same optimum.

In [3]:
from sklearn.cluster import KMeans

sk_km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_km)
print(f"sklearn KMeans inertia = {sk_km.inertia_:.4f} (ours: {trace[-1]:.4f})")
assert trace[-1] <= sk_km.inertia_ * 1.05, "our inertia much worse than sklearn"
print("PASS: within 5% of sklearn's inertia.")

sklearn KMeans inertia = 1881.0846 (ours: 1881.5669)
PASS: within 5% of sklearn's inertia.


## 3. PCA by eigendecomposition of the covariance (notes §9.1)

The data has a known low-dimensional structure: 500 points in 10 dimensions
generated as $X = Z A + \text{noise}$, with 3 latent factors $Z$, a random
$3 \times 10$ mixing matrix $A$, and noise standard deviation 0.3. PCA centers
the data, forms the sample covariance $\Sigma = X_c^\top X_c / (n-1)$, and takes
its eigenvectors sorted by eigenvalue. Expect the first 3 explained-variance
ratios to account for almost all of the variance, and the remaining 7 to be
tiny (they hold only the noise).

In [4]:
def pca_fit(X):
    # Returns (mean, components, eigenvalues); component rows are unit
    # eigenvectors sorted by decreasing eigenvalue.
    mean = X.mean(axis=0)
    Xc = X - mean
    cov = (Xc.T @ Xc) / (len(X) - 1)        # (d, d) sample covariance
    eigvals, eigvecs = np.linalg.eigh(cov)   # ascending order
    order = np.argsort(eigvals)[::-1]
    return mean, eigvecs[:, order].T, eigvals[order]


def pca_transform(X, mean, components, k):
    return (X - mean) @ components[:k].T


def pca_reconstruct(Z, mean, components, k):
    return Z @ components[:k] + mean


n, d, latent = 500, 10, 3
Z_true = np.random.randn(n, latent)
A = np.random.randn(latent, d) * 2.0
X_pca = Z_true @ A + 0.3 * np.random.randn(n, d)

mean, comps, evals = pca_fit(X_pca)
print("explained variance ratio by component:")
for i, v in enumerate(evals):
    print(f"  PC{i + 1:2d}: {v / evals.sum():.4f}")
print(f"first {latent} components together: {evals[:latent].sum() / evals.sum():.4f}")

explained variance ratio by component:
  PC 1: 0.5359
  PC 2: 0.2747
  PC 3: 0.1837
  PC 4: 0.0010
  PC 5: 0.0009
  PC 6: 0.0009
  PC 7: 0.0008
  PC 8: 0.0007
  PC 9: 0.0007
  PC10: 0.0007
first 3 components together: 0.9943


## 4. Check against sklearn and against the SVD (notes §9.3)

sklearn's `PCA` computes the SVD of the centered data. Its
`explained_variance_` should equal our eigenvalues, and its components should
equal ours up to sign (an eigenvector is only defined up to sign, so we
compare $|\cos|$ of each pair). The same eigenvalues should also come from our
own SVD of $X_c$ through $\lambda_j = s_j^2/(n-1)$.

In [5]:
from sklearn.decomposition import PCA

sk_pca = PCA().fit(X_pca)
ev_diff = float(np.max(np.abs(evals - sk_pca.explained_variance_)))
print(f"max |our eigenvalues - sklearn explained_variance_| = {ev_diff:.2e}")
assert ev_diff < 1e-8, "eigenvalues disagree with sklearn"

cos = np.abs(np.sum(comps * sk_pca.components_, axis=1))
print(f"min |cos(our PC, sklearn PC)| over components = {cos.min():.6f}")
assert np.all(cos > 1 - 1e-8), "component directions disagree with sklearn"
print("PASS: matches sklearn PCA (up to sign).")

s = np.linalg.svd(X_pca - mean, compute_uv=False)
svd_diff = float(np.max(np.abs(s ** 2 / (n - 1) - evals)))
print(f"max |s_j^2/(n-1) - lambda_j| = {svd_diff:.2e}")
assert svd_diff < 1e-8
print("PASS: eigenvalues of the covariance = squared singular values / (n-1).")

max |our eigenvalues - sklearn explained_variance_| = 2.13e-14
min |cos(our PC, sklearn PC)| over components = 1.000000
PASS: matches sklearn PCA (up to sign).
max |s_j^2/(n-1) - lambda_j| = 2.13e-14
PASS: eigenvalues of the covariance = squared singular values / (n-1).


## 5. Reconstruction error equals the tail sum of eigenvalues (notes §9.2)

Keep the top $k$ components, project, and map back. Notes §9.2 (and Exercise
11c) show that the total squared reconstruction error is
$\sum_i \|x_i - \hat x_i\|^2 = (n-1)\sum_{j>k}\lambda_j$. The loop prints both
sides for every $k$ and asserts they agree. It also prints the mean squared
error per entry: it should drop sharply up to $k = 3$, the number of latent
factors, and after that decrease only slowly, because each remaining
component holds only noise (about $0.3^2 = 0.09$ of variance per direction,
so roughly $0.09 \times (10-k)/10$ per entry).

In [6]:
print(f"{'k':>3} {'sum sq. error':>14} {'(n-1)*tail':>12} {'MSE/entry':>10}")
prev = np.inf
for k in range(1, d + 1):
    Xr = pca_reconstruct(pca_transform(X_pca, mean, comps, k), mean, comps, k)
    sse = float(np.sum((X_pca - Xr) ** 2))
    tail = (n - 1) * float(evals[k:].sum())
    mse_k = sse / (n * d)
    marker = "  <- number of latent factors" if k == latent else ""
    print(f"{k:>3} {sse:14.4f} {tail:12.4f} {mse_k:10.6f}{marker}")
    assert np.isclose(sse, tail, rtol=1e-8, atol=1e-8), "SSE != (n-1) * tail sum"
    assert mse_k <= prev + 1e-12, "reconstruction error should be non-increasing"
    prev = mse_k
print("PASS: reconstruction error = (n-1) * sum of discarded eigenvalues for every k.")

  k  sum sq. error   (n-1)*tail  MSE/entry
  1     27255.2358   27255.2358   5.451047
  2     11125.4215   11125.4215   2.225084
  3       334.9390     334.9390   0.066988  <- number of latent factors
  4       277.6707     277.6707   0.055534
  5       224.2020     224.2020   0.044840
  6       172.3400     172.3400   0.034468
  7       124.2732     124.2732   0.024855
  8        80.2855      80.2855   0.016057
  9        39.0246      39.0246   0.007805
 10         0.0000       0.0000   0.000000
PASS: reconstruction error = (n-1) * sum of discarded eigenvalues for every k.
